In [5]:
import xml.etree.ElementTree as ET
import pandas as pd
import re
import numpy as np

In [6]:
kml_file = '../src/satpass/data/nominal_tracks/swot_science_orbit_feb2025-v1.0_10s_nadir.kml'
csv_file = '../src/satpass/data/nominal_tracks/swot_science_orbit_feb2025-v1.0_10s_nadir.csv'

In [7]:
tree = ET.parse(kml_file)
root = tree.getroot()

ns = {'kml': 'http://www.opengis.net/kml/2.2'}

rows = []

# %%
for placemark in root.findall('.//kml:Placemark', ns):

    name_node = placemark.find('kml:name', ns)
    description_node = placemark.find('kml:description', ns)

    name = name_node.text if name_node is not None else ''
    description = description_node.text if description_node is not None else ''

    # Extract pass number from placemark name
    match = re.search(r'(\d+)', name)

    if match is not None:
        pass_number = int(match.group(1))
    else:
        pass_number = None

    # There can be several LineStrings / coordinate blocks per placemark
    coord_nodes = placemark.findall('.//kml:coordinates', ns)

    for segment_id, coord_node in enumerate(coord_nodes):

        coordinates = coord_node.text.strip().split()

        for coordinate_index, coord in enumerate(coordinates):

            values = coord.split(',')

            longitude = float(values[0])
            latitude = float(values[1])
            altitude = float(values[2]) if len(values) > 2 else np.nan

            rows.append({
                'pass_number': pass_number,
                'segment_id': segment_id,
                'description': description,
                'coordinate_index': coordinate_index,
                'longitude': longitude,
                'latitude': latitude,
                'altitude': altitude
            })

# %%
orbit = pd.DataFrame(rows)

orbit

,pass_number,segment_id,description,coordinate_index,longitude,latitude,altitude
0,1,0,Day 01 00:00:00,0,-83.378233,-77.674650,NaN
1,1,0,Day 01 00:00:00,1,-80.711618,-77.661530,NaN
2,1,0,Day 01 00:00:00,2,-78.056370,-77.621778,NaN
3,1,0,Day 01 00:00:00,3,-75.423693,-77.555650,NaN
4,1,0,Day 01 00:00:00,4,-72.824222,-77.463562,NaN
...,...,...,...,...,...,...,...
181391,584,0,Day 21 19:53:40,245,-95.266883,-77.403095,NaN
181392,584,0,Day 21 19:53:40,246,-92.689739,-77.509104,NaN
181393,584,0,Day 21 19:53:40,247,-90.073872,-77.589362,NaN
181394,584,0,Day 21 19:53:40,248,-87.429518,-77.643368,NaN


In [9]:
orbit[['pass_number', 'longitude', 'latitude']].to_csv(csv_file, index=False)